# Experiment 6: GloVe Word Embeddings and Statistical N-gram Language Model

This notebook implements:
1. **GloVe (Global Vectors for Word Representation) from scratch** (without any pretrained model) on `glove_corpus.txt` to generate word embeddings and find the most similar words to `"King"`.
2. **Pretrained GloVe Word Embeddings using Gensim** (`glove-wiki-gigaword-50`) to find the most similar words to `"King"`.
3. **Statistical N-gram Language Model from scratch** (Unigram, Bigram, and Trigram with backoff) to predict the next word in a sequence based on preceding words.

## Part 1.1: GloVe from Scratch (Without Using Any Pretrained Model)

We build a symmetric word-word co-occurrence matrix $X$ from `glove_corpus.txt` with inverse-distance weighting within a context window, then train word vectors $W, \tilde{W}$ and biases $b, \tilde{b}$ using AdaGrad to minimize the GloVe objective:

$$J = \sum_{i,j:\, X_{ij} > 0} f(X_{ij}) \left( w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2$$

where $f(x) = \min\left(1, (x / x_{\max})^\alpha\right)$.

In [1]:
import re
from collections import Counter
from pathlib import Path
import numpy as np

# Load corpus
corpus_path = Path("glove_corpus.txt")
raw_text = corpus_path.read_text(encoding="utf-8")
sentences = [
    re.findall(r"[a-z]+", line.lower())
    for line in raw_text.splitlines()
    if line.strip()
]

# Build vocabulary
word_counts = Counter(word for sent in sentences for word in sent)
vocab = sorted(word_counts.keys())
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)

print(f"Loaded {len(sentences)} sentences from {corpus_path.name}")
print(f"Total tokens: {sum(word_counts.values())}, Vocabulary size: {V}")

Loaded 24 sentences from glove_corpus.txt
Total tokens: 148, Vocabulary size: 68


In [2]:
# Build symmetric co-occurrence matrix with 1/distance weighting
window_size = 4
cooc = np.zeros((V, V), dtype=np.float64)

for sent in sentences:
    indices = [word2idx[w] for w in sent]
    for i, center_id in enumerate(indices):
        start = max(0, i - window_size)
        end = min(len(indices), i + window_size + 1)
        for j in range(start, end):
            if i == j:
                continue
            dist = abs(i - j)
            cooc[center_id, indices[j]] += 1.0 / dist

nonzero_pairs = np.argwhere(cooc > 0)
print(f"Non-zero co-occurrence pairs: {len(nonzero_pairs)}")

Non-zero co-occurrence pairs: 496


In [3]:
# Train GloVe embeddings from scratch using AdaGrad
def train_glove(cooc_matrix, nonzero_indices, vocab_size, embed_dim=50,
                x_max=10.0, alpha=0.75, lr=0.05, epochs=400, seed=42):
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.1, size=(vocab_size, embed_dim))
    W_tilde = rng.normal(0, 0.1, size=(vocab_size, embed_dim))
    b = np.zeros(vocab_size, dtype=np.float64)
    b_tilde = np.zeros(vocab_size, dtype=np.float64)

    grad_sq_W = np.ones_like(W)
    grad_sq_W_tilde = np.ones_like(W_tilde)
    grad_sq_b = np.ones_like(b)
    grad_sq_b_tilde = np.ones_like(b_tilde)

    loss_history = []
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        rng.shuffle(nonzero_indices)
        for i, j in nonzero_indices:
            x_ij = cooc_matrix[i, j]
            weight = (x_ij / x_max) ** alpha if x_ij < x_max else 1.0
            diff = np.dot(W[i], W_tilde[j]) + b[i] + b_tilde[j] - np.log(x_ij)
            total_loss += 0.5 * weight * (diff ** 2)

            grad_common = weight * diff
            grad_W_i = grad_common * W_tilde[j]
            grad_W_tilde_j = grad_common * W[i]

            grad_sq_W[i] += grad_W_i ** 2
            grad_sq_W_tilde[j] += grad_W_tilde_j ** 2
            grad_sq_b[i] += grad_common ** 2
            grad_sq_b_tilde[j] += grad_common ** 2

            W[i] -= (lr / np.sqrt(grad_sq_W[i])) * grad_W_i
            W_tilde[j] -= (lr / np.sqrt(grad_sq_W_tilde[j])) * grad_W_tilde_j
            b[i] -= (lr / np.sqrt(grad_sq_b[i])) * grad_common
            b_tilde[j] -= (lr / np.sqrt(grad_sq_b_tilde[j])) * grad_common

        avg_loss = total_loss / len(nonzero_indices)
        loss_history.append(avg_loss)
        if epoch % 100 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.6f}")

    # Combine main and context word vectors
    embeddings = W + W_tilde
    return embeddings, loss_history

embeddings, loss_history = train_glove(cooc, nonzero_pairs.copy(), V)

Epoch   1/400 | Loss: 0.043873
Epoch 100/400 | Loss: 0.001932
Epoch 200/400 | Loss: 0.000184
Epoch 300/400 | Loss: 0.000027
Epoch 400/400 | Loss: 0.000005


In [4]:
# Find most similar words to "King" using cosine similarity
def most_similar_scratch(query_word, embeddings, word2idx, idx2word, top_n=10):
    key = query_word.lower()
    if key not in word2idx:
        raise KeyError(f"'{query_word}' not found in vocabulary.")

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normed = embeddings / np.maximum(norms, 1e-12)

    q_idx = word2idx[key]
    sims = normed @ normed[q_idx]
    ranked_indices = np.argsort(-sims)

    results = []
    for idx in ranked_indices:
        if idx == q_idx:
            continue
        results.append((idx2word[idx], float(sims[idx])))
        if len(results) >= top_n:
            break
    return results

target_word = "King"
similar_scratch = most_similar_scratch(target_word, embeddings, word2idx, idx2word, top_n=10)

print(f"Top 10 most similar words to '{target_word}' (GloVe from scratch):")
print(f"{'Rank':<6}{'Word':<15}{'Cosine Similarity'}")
print("-" * 38)
for rank, (word, score) in enumerate(similar_scratch, start=1):
    print(f"{rank:<6}{word:<15}{score:.4f}")

Top 10 most similar words to 'King' (GloVe from scratch):
Rank  Word           Cosine Similarity
--------------------------------------
1     the            0.6670
2     lives          0.3734
3     queen          0.3655
4     rules          0.3422
5     is             0.3157
6     and            0.2577
7     prince         0.2405
8     princess       0.2110
9     develops       0.2045
10    brave          0.1641


## Part 1.2: GloVe Using Pretrained Model (Gensim)

Now we load the pretrained `glove-wiki-gigaword-50` embeddings (trained on Wikipedia + Gigaword, 400,000 vocabulary words, 50 dimensions) via `gensim.downloader` and query the most similar words to `"King"` (`"king"`).

In [5]:
import gensim.downloader as api

pretrained_glove = api.load("glove-wiki-gigaword-50")
print(f"Loaded pretrained GloVe model with {len(pretrained_glove)} words (dim={pretrained_glove.vector_size}).")

target_word = "King"
similar_pretrained = pretrained_glove.most_similar(target_word.lower(), topn=10)

print(f"\nTop 10 most similar words to '{target_word}' (Pretrained Gensim GloVe):")
print(f"{'Rank':<6}{'Word':<15}{'Cosine Similarity'}")
print("-" * 38)
for rank, (word, score) in enumerate(similar_pretrained, start=1):
    print(f"{rank:<6}{word:<15}{score:.4f}")

# Classic analogy verification: king - man + woman ≈ ?
analogy = pretrained_glove.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
print("\nAnalogy test (king - man + woman):", [(w, round(s, 4)) for w, s in analogy])

Loaded pretrained GloVe model with 400000 words (dim=50).

Top 10 most similar words to 'King' (Pretrained Gensim GloVe):
Rank  Word           Cosine Similarity
--------------------------------------
1     prince         0.8236
2     queen          0.7839
3     ii             0.7746
4     emperor        0.7736
5     son            0.7667
6     uncle          0.7627
7     kingdom        0.7542
8     throne         0.7540
9     brother        0.7492
10    ruler          0.7434

Analogy test (king - man + woman): [('queen', 0.8524), ('throne', 0.7664), ('prince', 0.7592)]


## Part 2: Statistical N-gram Language Model from Scratch

We implement a statistical N-gram language model from scratch (no external NLP toolkits) on the provided corpus.
- Sentences are split at `.` boundaries and padded with `<BOS>` tokens so context probabilities respect sentence starts.
- We compute **Unigram**, **Bigram**, and **Trigram** counts and conditional Maximum Likelihood Estimation (MLE) probabilities:
  $$P(w_i \mid w_{i-n+1}, \dots, w_{i-1}) = \frac{C(w_{i-n+1}, \dots, w_i)}{C(w_{i-n+1}, \dots, w_{i-1})}$$
- For next-word prediction, we check the highest matching N-gram order (Trigram $\rightarrow$ Bigram $\rightarrow$ Unigram backoff).

In [6]:
from collections import defaultdict, Counter

ngram_corpus = (
    "natural language processing is a branch of artificial intelligence."
    "natural language processing enables computers to understand human language."
    "machine learning is a branch of artificial intelligence."
    "deep learning is a part of machine learning."
    "language models learn patterns from text data."
    "a language model predicts the next word in a sentence."
    "natural language processing uses language models."
    "artificial intelligence includes machine learning and deep learning."
)

# Sentence segmentation and tokenization
raw_sentences = [s.strip() for s in ngram_corpus.split(".") if s.strip()]
tokenized_sentences = [s.lower().split() for s in raw_sentences]

vocab_ngram = sorted({w for sent in tokenized_sentences for w in sent})
print(f"Number of sentences: {len(tokenized_sentences)}")
print(f"Total word tokens:   {sum(len(s) for s in tokenized_sentences)}")
print(f"Vocabulary size:     {len(vocab_ngram)}")

Number of sentences: 8
Total word tokens:   65
Vocabulary size:     34


In [7]:
class NGramLanguageModel:
    """Statistical N-gram language model built from scratch with backoff."""

    def __init__(self, tokenized_sents, max_n=3):
        self.max_n = max_n
        self.counts = {n: defaultdict(Counter) for n in range(1, max_n + 1)}
        self.context_totals = {n: Counter() for n in range(1, max_n + 1)}
        self.total_unigrams = 0
        self._fit(tokenized_sents)

    def _fit(self, tokenized_sents):
        for sent in tokenized_sents:
            for w in sent:
                self.counts[1][()][w] += 1
                self.total_unigrams += 1

            for n in range(2, self.max_n + 1):
                padded = ["<BOS>"] * (n - 1) + sent
                for i in range(n - 1, len(padded)):
                    ctx = tuple(padded[i - (n - 1):i])
                    nxt = padded[i]
                    self.counts[n][ctx][nxt] += 1
                    self.context_totals[n][ctx] += 1

    def predict_next(self, sequence, top_k=3):
        tokens = sequence.lower().strip().split()
        # Try highest N-gram order down to 1 (backoff)
        for n in range(min(self.max_n, len(tokens) + 1), 1, -1):
            ctx = tuple(tokens[-(n - 1):])
            if ctx in self.counts[n]:
                total = self.context_totals[n][ctx]
                candidates = [
                    (word, count / total)
                    for word, count in self.counts[n][ctx].most_common(top_k)
                ]
                return n, ctx, candidates

        # Unigram fallback
        candidates = [
            (word, count / self.total_unigrams)
            for word, count in self.counts[1][()].most_common(top_k)
        ]
        return 1, (), candidates

    def generate_text(self, seed_text, num_words=6):
        tokens = seed_text.lower().strip().split()
        for _ in range(num_words):
            _, _, candidates = self.predict_next(" ".join(tokens), top_k=1)
            if not candidates:
                break
            tokens.append(candidates[0][0])
        return " ".join(tokens)


lm = NGramLanguageModel(tokenized_sentences, max_n=3)

# Show sample Trigram and Bigram probabilities
print("Sample Trigram probabilities for ('natural', 'language'):")
for word, cnt in lm.counts[3][("natural", "language")].most_common():
    total = lm.context_totals[3][("natural", "language")]
    print(f"  P({word} | natural language) = {cnt}/{total} = {cnt/total:.4f}")

print("\nSample Bigram probabilities for ('machine',):")
for word, cnt in lm.counts[2][("machine",)].most_common():
    total = lm.context_totals[2][("machine",)]
    print(f"  P({word} | machine) = {cnt}/{total} = {cnt/total:.4f}")

Sample Trigram probabilities for ('natural', 'language'):
  P(processing | natural language) = 3/3 = 1.0000

Sample Bigram probabilities for ('machine',):
  P(learning | machine) = 3/3 = 1.0000


In [8]:
# Test next-word prediction on multiple sequences
test_sequences = [
    "natural language",
    "natural language processing",
    "a branch of",
    "machine learning is",
    "deep learning is a",
    "predicts the next",
    "artificial intelligence",
    "language models",
]

print(f"{'Input Sequence':<30}{'Order':<8}{'Predicted Word':<18}{'Top Candidates (word: prob)'}")
print("-" * 92)
for seq in test_sequences:
    order, ctx, preds = lm.predict_next(seq, top_k=3)
    best_word = preds[0][0]
    cand_str = ", ".join(f"{w}: {p:.3f}" for w, p in preds)
    print(f"{seq:<30}{str(order)+'-gram':<8}{best_word:<18}{cand_str}")

print("\nMulti-word Sequence Generation:")
for seed in ["natural language", "machine learning", "a language model"]:
    print(f"  Seed: '{seed}' -> '{lm.generate_text(seed, num_words=5)}'")

Input Sequence                Order   Predicted Word    Top Candidates (word: prob)
--------------------------------------------------------------------------------------------
natural language              3-gram  processing        processing: 1.000
natural language processing   3-gram  is                is: 0.333, enables: 0.333, uses: 0.333
a branch of                   3-gram  artificial        artificial: 1.000
machine learning is           3-gram  a                 a: 1.000
deep learning is a            3-gram  branch            branch: 0.667, part: 0.333
predicts the next             3-gram  word              word: 1.000
artificial intelligence       3-gram  includes          includes: 1.000
language models               3-gram  learn             learn: 1.000

Multi-word Sequence Generation:
  Seed: 'natural language' -> 'natural language processing is a branch of'
  Seed: 'machine learning' -> 'machine learning is a branch of artificial'
  Seed: 'a language model' -> 'a languag